<a href="https://colab.research.google.com/github/akarshitbaliyan-byte/pyhton/blob/main/Akarshit2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""
SHOP SALES ANALYSIS
------------------------------------------------
Libraries used:
1. Pandas
2. NumPy

Topics used:
- Variables and data types
- Functions
- Pandas DataFrames
- NumPy arrays and functions
- Basic OOP (class and object)
- Filtering, grouping, sorting and summary statistics
"""

import pandas as pd
import numpy as np


def create_sample_data():

    data = {
        "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
                    1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020],

        "Date": [
            "2026-01-03", "2026-01-04", "2026-01-05", "2026-01-08",
            "2026-01-10", "2026-01-12", "2026-01-15", "2026-01-20",
            "2026-01-25", "2026-02-02", "2026-02-04", "2026-02-07",
            "2026-02-10", "2026-02-14", "2026-02-18", "2026-02-22",
            "2026-03-02", "2026-03-05", "2026-03-09", "2026-03-14"
        ],

        "Product": [
            "Notebook", "Pen", "Backpack", "Notebook", "Water Bottle",
            "Pen", "Pencil Box", "Calculator", "Backpack", "Backpack",
            "Water Bottle", "Notebook", "Pen", "Backpack", "Water Bottle",
            "Pencil Box", "Calculator", "Notebook", "Pen", "Water Bottle"
        ],

        "Category": [
            "Stationery", "Stationery", "Accessories", "Stationery", "Accessories",
            "Stationery", "Stationery", "Accessories", "Accessories", "Accessories",
            "Accessories", "Stationery", "Stationery", "Accessories", "Accessories",
            "Stationery", "Accessories", "Stationery", "Stationery", "Accessories"
        ],

        "Quantity": [5, 10, 2, 8, 3, 15, 4, 2, 1, 4,
                     6, 7, 12, 3, 5, 6, 3, 10, 20, 4],

        "UnitPrice": [50, 10, 800, 50, 250, 10, 120, 350, 800, 800,
                      250, 50, 10, 800, 250, 120, 350, 50, 10, 250],

        "Region": [
            "North", "South", "East", "West", "North",
            "South", "East", "West", "North", "East",
            "West", "North", "South", "East", "West",
            "South", "North", "East", "West", "South"
        ]
    }

    return pd.DataFrame(data)


class RetailSalesAnalyzer:

    def __init__(self, dataframe):
        self.df = dataframe.copy()
        self.prepare_data()

    def prepare_data(self):

        self.df["Date"] = pd.to_datetime(self.df["Date"], errors="coerce")
        self.df["Quantity"] = pd.to_numeric(self.df["Quantity"], errors="coerce")
        self.df["UnitPrice"] = pd.to_numeric(self.df["UnitPrice"], errors="coerce")

        # remove rows which have missing values
        self.df = self.df.dropna(
            subset=["Date", "Quantity", "UnitPrice",
                    "Product", "Category", "Region"]
        )

        # sales = quantity * price
        self.df["Sales"] = np.multiply(
            self.df["Quantity"].to_numpy(),
            self.df["UnitPrice"].to_numpy()
        )

        self.df["Month"] = self.df["Date"].dt.to_period("M").astype(str)

        # mark each order as High or Low using the average sales
        average = np.mean(self.df["Sales"].to_numpy())
        self.df["OrderType"] = np.where(self.df["Sales"] > average, "High", "Low")

    def show_basic_info(self):

        print("\n--- DATA OVERVIEW ---")
        print("Number of records:", len(self.df))
        print("Number of products:", self.df["Product"].nunique())
        print("Categories:", ", ".join(sorted(self.df["Category"].unique())))
        print("Date range:", self.df["Date"].min().date(),
              "to", self.df["Date"].max().date())

        print("\nFirst five records:")
        print(self.df.head().to_string(index=False))

    def total_sales(self):
        return np.sum(self.df["Sales"].to_numpy())

    def average_sales(self):
        return np.mean(self.df["Sales"].to_numpy())

    def median_sales(self):
        return np.median(self.df["Sales"].to_numpy())

    def std_sales(self):
        return np.std(self.df["Sales"].to_numpy())

    def highest_order(self):
        # row with the maximum sales
        return self.df.loc[self.df["Sales"].idxmax()]

    def lowest_order(self):
        return self.df.loc[self.df["Sales"].idxmin()]

    def sales_by_category(self):
        return (
            self.df.groupby("Category")["Sales"]
            .sum()
            .sort_values(ascending=False)
        )

    def category_share(self):
        # percentage of total sales for each category
        sales = self.sales_by_category()
        return (sales / sales.sum() * 100).round(2)

    def sales_by_region(self):
        return (
            self.df.groupby("Region")["Sales"]
            .sum()
            .sort_values(ascending=False)
        )

    def top_products(self, count=5):
        return (
            self.df.groupby("Product")["Sales"]
            .sum()
            .sort_values(ascending=False)
            .head(count)
        )

    def monthly_sales(self):
        return (
            self.df.groupby("Month")["Sales"]
            .sum()
            .sort_index()
        )

    def monthly_growth(self):
        # change in sales compared to the previous month (in %)
        return (self.monthly_sales().pct_change() * 100).round(2)

    def best_selling_products_by_quantity(self, count=5):
        return (
            self.df.groupby("Product")["Quantity"]
            .sum()
            .sort_values(ascending=False)
            .head(count)
        )

    def top_orders(self, count=3):
        return (
            self.df.sort_values("Sales", ascending=False)
            .head(count)[["OrderID", "Product", "Region", "Sales"]]
        )

    def high_value_orders(self, limit=2000):
        # filtering orders above the given limit
        return self.df[self.df["Sales"] > limit][["OrderID", "Product", "Sales"]]

    def region_category_table(self):
        return pd.pivot_table(
            self.df,
            values="Sales",
            index="Region",
            columns="Category",
            aggfunc="sum",
            fill_value=0
        )

    def order_type_count(self):
        return self.df["OrderType"].value_counts()

    def save_summary(self):

        self.sales_by_category().to_csv("sales_by_category.csv", header=["Sales"])
        self.sales_by_region().to_csv("sales_by_region.csv", header=["Sales"])
        self.top_products().to_csv("top_products.csv", header=["Sales"])
        self.monthly_sales().to_csv("monthly_sales.csv", header=["Sales"])

        print("\nSummary files saved in the project folder.")


def load_data():

    try:
        dataframe = pd.read_csv("retail_sales.csv")
        print("Loaded data from retail_sales.csv")

    except FileNotFoundError:
        print("retail_sales.csv not found. Using sample data.")
        dataframe = create_sample_data()

    required_columns = {
        "OrderID", "Date", "Product", "Category",
        "Quantity", "UnitPrice", "Region"
    }

    missing = required_columns - set(dataframe.columns)

    if missing:
        raise ValueError(
            "Missing required CSV columns: " + ", ".join(sorted(missing))
        )

    return dataframe


def main():

    data = load_data()
    analyzer = RetailSalesAnalyzer(data)

    analyzer.show_basic_info()

    print("\n--- SALES STATISTICS ---")
    print("Total Sales   : Rs.", analyzer.total_sales())
    print("Average Sales : Rs.", round(analyzer.average_sales(), 2))
    print("Median Sales  : Rs.", analyzer.median_sales())
    print("Std Deviation : Rs.", round(analyzer.std_sales(), 2))

    high = analyzer.highest_order()
    low = analyzer.lowest_order()
    print("Highest order : Order", high["OrderID"], "-", high["Product"],
          "- Rs.", high["Sales"])
    print("Lowest order  : Order", low["OrderID"], "-", low["Product"],
          "- Rs.", low["Sales"])

    print("\n--- SALES BY CATEGORY ---")
    print(analyzer.sales_by_category())

    print("\n--- CATEGORY SHARE (%) ---")
    print(analyzer.category_share())

    print("\n--- SALES BY REGION ---")
    print(analyzer.sales_by_region())

    print("\n--- REGION vs CATEGORY ---")
    print(analyzer.region_category_table())

    print("\n--- TOP PRODUCTS BY SALES ---")
    print(analyzer.top_products())

    print("\n--- BEST SELLING PRODUCTS BY QUANTITY ---")
    print(analyzer.best_selling_products_by_quantity())

    print("\n--- MONTHLY SALES ---")
    print(analyzer.monthly_sales())

    print("\n--- MONTHLY GROWTH (%) ---")
    print(analyzer.monthly_growth())

    print("\n--- TOP 3 ORDERS ---")
    print(analyzer.top_orders().to_string(index=False))

    print("\n--- ORDERS ABOVE Rs. 2000 ---")
    print(analyzer.high_value_orders().to_string(index=False))

    print("\n--- HIGH / LOW ORDERS ---")
    print(analyzer.order_type_count())

    analyzer.save_summary()


if __name__ == "__main__":
    main()

Loaded data from retail_sales.csv

--- DATA OVERVIEW ---
Number of records: 20
Number of products: 6
Categories: Accessories, Stationery
Date range: 2026-01-03 to 2026-03-14

First five records:
 OrderID       Date      Product    Category  Quantity  UnitPrice Region  Sales   Month OrderType
    1001 2026-01-03     Notebook  Stationery         5         50  North    250 2026-01       Low
    1002 2026-01-04          Pen  Stationery        10         10  South    100 2026-01       Low
    1003 2026-01-05     Backpack Accessories         2        800   East   1600 2026-01      High
    1004 2026-01-08     Notebook  Stationery         8         50   West    400 2026-01       Low
    1005 2026-01-10 Water Bottle Accessories         3        250  North    750 2026-01       Low

--- SALES STATISTICS ---
Total Sales   : Rs. 17520
Average Sales : Rs. 876.0
Median Sales  : Rs. 710.0
Std Deviation : Rs. 782.65
Highest order : Order 1010 - Backpack - Rs. 3200
Lowest order  : Order 1002 - Pen - Rs